In [1]:
import k_fail_MBTR
import BB_wrapper
import k_fail_prediction

import torch

/Users/karim/projects/k-sPSS/.venv/lib/python3.11/site-packages/pycutest/__init__.py:26: RuntimeWarning: the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.
  warnings.warn("the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.", RuntimeWarning)


# Wrapper

In [2]:
cutest_wrapper = BB_wrapper.BB_cutest_collection(write_to_file="cutest_problem_selection.txt", max_dim=100, cap_n_problems=1)

loading problems...
0/276
loaded 1 problems


In [3]:
# evaluating n problem functions
for i in range(1):
    p = cutest_wrapper.problems[i]
    f = cutest_wrapper.problem_functions[i]
    print(f"{p.name} | n: {p.n}: {f(torch.from_numpy(p.x0))}")

SISSER | n: 2: 3.02030030003


# MBTR

In [9]:
problem_idx = 0
k = 2
k_fail_wrapper = BB_wrapper.BB_k_fail_wrapper(cutest_wrapper.problem_functions[problem_idx], k*torch.ones((1024, 2), dtype=torch.int16))

alg = k_fail_MBTR.MBTR_k_fail(torch.from_numpy(cutest_wrapper.problems[problem_idx].x0), k_fail_wrapper, 1, 1e-1, 0.1, 0.5, 1e-1, k_fail_prediction.constant_prediction_software(k), log_file_path="alg_logs/MBTR/testing.txt")

In [10]:
alg.log_current()
for i in range(1000):
    if alg.step_default():
        break


# is n + 2k + 1 fair?
# with it, always linear: 1847 function evals
# with always quad: 448 function evals

In [7]:
# H: None
# points: [[ 0.          0.        ]
#  [-0.15244442  0.4307067 ]
#  [-0.05507779  0.22729501]
#  [-0.39858192  0.6989344 ]
#  [-0.04768276  0.20510971]
#  [-0.0833317   0.29964896]]

# fixed version (still breaks)
# H: None
# points: [[0.         0.        ]
#  [0.21817741 0.44988734]
#  [0.45346695 0.21063653]
#  [0.45938012 0.19740793]
#  [0.41485953 0.27909064]
#  [0.49603891 0.06281256]]


broken_x_k = torch.tensor([1.0000, 0.1000])
broken_points = torch.tensor([
 [0.        , 0.        ],
 [0.21817741, 0.44988734],
 [0.45346695, 0.21063653],
 [0.45938012, 0.19740793],
 [0.41485953, 0.27909064],
 [0.49603891, 0.06281256]])

broken_func_vals = []

for p in broken_points:
    print(p, " | ", cutest_wrapper.problem_functions[0](p + broken_x_k))
    broken_func_vals.append(cutest_wrapper.problem_functions[0](p + broken_x_k))

broken_func_vals = torch.tensor(broken_func_vals)


tensor([0., 0.])  |  3.0203003006439277
tensor([0.2182, 0.4499])  |  7.77809909471623
tensor([0.4535, 0.2106])  |  13.824444939862795
tensor([0.4594, 0.1974])  |  14.008260278544622
tensor([0.4149, 0.2791])  |  12.659262761617814
tensor([0.4960, 0.0628])  |  15.148476370717711


In [11]:
import models

models.get_quad_model_and_solution(broken_points, broken_func_vals, 0.25)

status: optimal
H: [[-3.54161607 -2.96036918]
 [-2.96036918 -4.42442909]]
points: [[0.         0.        ]
 [0.21817741 0.44988734]
 [0.45346695 0.21063653]
 [0.45938012 0.19740793]
 [0.41485953 0.27909064]
 [0.4960389  0.06281256]]


(tensor([-0.1599, -0.1922], dtype=torch.float64),
 -0.49873078674542165,
 tensor([1.2557, 1.5506], dtype=torch.float64),
 <function models.get_quad_model_and_solution.<locals>.f_tilda(x)>)

In [6]:
broken_func_vals

tensor([0.0000, 0.1490, 0.1510, 0.1546, 0.1339, 0.1836])

In [9]:
torch.linspace(1, 6, 6)[torch.tensor([1, 2, 3, 4, 5, 0], dtype=torch.int32)]

tensor([2., 3., 4., 5., 6., 1.])